# Fine-Tuning Laya (multilingual checkpoint) no Colab, 1x GPU

Adaptado do notebook oficial (`NandhaKishorM/laya`, Kaggle 2xT4 DDP) para **Colab de 1 GPU** — sem `torch.distributed`, sem DDP, mesmo algoritmo de treino (RLCD).

Baixa só o subfolder `multilingual` do repo `convaiinnovations/laya` (o `laya_provider.py` original apontava pro checkpoint em inglês por engano — aqui já corrigido).

**No Colab:** Ambiente de execução -> Alterar tipo de ambiente de execução -> GPU (T4 é suficiente).

**Seu dataset:** gere com `python to_laya_jsonl.py` (pasta `ptbr-intent-nli`) e suba `train.jsonl` e `test.jsonl` em `/content/`. Uma linha por pedido, uma pergunta `choice`:
```json
{"state": "explica o ciclo de Krebs e depois me faz 5 questões", "questions": {"geracao": {"type": "choice", "instructions": "Que tipo de material de estudo o usuário está pedindo para gerar?", "criteria": {"flashcard": "...", "questionario": "...", "aula": "...", "outro": "..."}}}, "gold": {"geracao": {"label": "questionario", "probabilities": {"flashcard": 0.0, "questionario": 0.5, "aula": 0.5, "outro": 0.0}}}}
```
Multi-intenção = massa dividida entre os rótulos; fora do escopo = `outro`.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Ative GPU em Ambiente de execução -> Alterar tipo de ambiente de execução"
print(torch.cuda.get_device_name(0))


## 2. Instalar dependências

In [ ]:
!pip install -q -U "laya>=0.3.11" "transformers>=4.48.0" safetensors huggingface_hub
import laya, transformers, torch
print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)


## 3. Baixar só o checkpoint multilingual + preparar seu dataset

In [ ]:
import os, json
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
SUBFOLDER = "multilingual"  # bug do laya_provider.py original: apontava pro repo raiz (inglês)

bundle_dir = snapshot_download(
    MODEL_ID,
    allow_patterns=[SUBFOLDER + "/" + n for n in ("rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*")],
)
model_dir = os.path.join(bundle_dir, SUBFOLDER)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

DATASET_PATH = "/content/train.jsonl"  # suba seu arquivo antes de rodar esta célula
assert os.path.exists(DATASET_PATH), f"Suba seu dataset em {DATASET_PATH} primeiro (painel Arquivos, à esquerda)"

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}

items = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        state, questions, gold = row["state"], row["questions"], row["gold"]
        for qid, q in questions.items():
            if qid in gold:
                it = build_training_item(state, q, gold[qid])
                if it:
                    items.append(it)

print(f"{len(items)} sequências de treino preparadas a partir de {DATASET_PATH}")
assert len(items) >= 20, "Poucos exemplos — o script separa uma fatia de calibração; use pelo menos ~50 casos"


## 4. Treino RLCD (1 GPU, sem DDP)

In [ ]:
import random, time
from safetensors.torch import save_file
from laya.common import build_model, proper_reward

OUTPUT_DIR = "/content/laya_finetuned"
device = torch.device("cuda")

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, :len(it["ids"])] = torch.tensor(it["ids"])
        att[i, :len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, :len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
            "target": target, "qtype": torch.tensor([it["qtype"] for it in items]),
            "label": torch.tensor([it["label"] for it in items])}

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

cfg["gradient_checkpointing"] = True
model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
from safetensors.torch import load_file
model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.head_checkpointing = True
model.to(device)
model.train()

# Fatia fixa de calibração, nunca entra em batch de treino (mesmo critério do script oficial)
CALIB_MAX = 400
order = list(range(len(items)))
random.Random(20260922).shuffle(order)
n_calib = min(CALIB_MAX, max(1, len(items) // 10))
calib_items = [items[i] for i in sorted(order[:n_calib])]
train_items = [items[i] for i in sorted(order[n_calib:])]

EPOCHS = 4
MICRO_BATCH = 8       # baixe se der OOM na T4 (ex.: 4)
GRAD_ACCUM = 8         # efetivo = MICRO_BATCH * GRAD_ACCUM = 64, igual ao script oficial (2 GPUs x 4)
GROUP_SIZE = 4
LR_ENCODER = 2.5e-5
LR_HEAD = 1.0e-4
SIGMA_START, SIGMA_END = 0.4, 0.1

enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
optimizer = torch.optim.AdamW(
    [{"params": enc_params, "lr": LR_ENCODER}, {"params": head_params, "lr": LR_HEAD}],
    weight_decay=0.01,
)
total_updates = max(1, (len(train_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_updates, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=True)

print(f"Treinando: {len(train_items)} itens ({len(calib_items)} reservados pra calibração)")
t0 = time.time()
for epoch in range(EPOCHS):
    random.seed(42 + epoch)
    random.shuffle(train_items)
    epoch_loss, n_batches, accum_step = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)
    sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * (epoch / max(1, EPOCHS - 1))

    for b_idx in range(0, len(train_items), MICRO_BATCH):
        chunk = train_items[b_idx:b_idx + MICRO_BATCH]
        if not chunk:
            continue
        batch = collate_train_batch(chunk, tok.pad_token_id)
        with torch.autocast("cuda", dtype=torch.float16):
            logits, act = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                                 batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                                 batch["qtype"].to(device))
        logits = logits.float()
        mask = batch["marker_mask"].to(device)
        k = mask.sum(-1, keepdim=True).float()
        target = batch["target"].to(device)

        eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
        eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
        z = logits.detach().unsqueeze(0) + eps
        q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
        with torch.no_grad():
            r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
            adv = r - r.mean(0, keepdim=True)
            adv = adv / (adv.std() + 1e-6)
        logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
        loss_rl = -(adv * logp).mean()
        loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
        loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()

        scaler.scale(loss).backward()
        accum_step += 1
        if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(train_items):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        epoch_loss += loss.item() * GRAD_ACCUM
        n_batches += 1
        if n_batches % 10 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss {loss.item()*GRAD_ACCUM:.4f} | Reward {r.mean().item():.3f} | LR {scheduler.get_last_lr()[0]:.2e}")

    print(f"=== Epoch {epoch+1}/{EPOCHS} em {time.time()-t0:.1f}s | Loss médio {epoch_loss/max(1,n_batches):.4f} ===")
    ckpt_dir = os.path.join(OUTPUT_DIR, "checkpoint_latest")
    os.makedirs(ckpt_dir, exist_ok=True)
    save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}, os.path.join(ckpt_dir, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
    tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
    print(f"  checkpoint salvo em {ckpt_dir}")


## 5. Calibração de temperatura + salvar modelo final

In [ ]:
print("Ajustando temperaturas de calibração...")
del optimizer, scaler, scheduler
torch.cuda.empty_cache()
model.eval()
calib_preds = []
with torch.no_grad():
    for c_idx in range(0, len(calib_items), 16):
        c_chunk = calib_items[c_idx:c_idx + 16]
        cb = collate_train_batch(c_chunk, tok.pad_token_id)
        with torch.autocast("cuda", dtype=torch.float16):
            l_sub, _ = model(cb["input_ids"].to(device), cb["attention_mask"].to(device),
                              cb["marker_pos"].to(device), cb["marker_mask"].to(device),
                              cb["qtype"].to(device))
        l_np = l_sub.float().cpu().numpy()
        for r, it in enumerate(c_chunk):
            k = len(it["markers"])
            calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))

fitted_temps = [1.2, 1.2, 1.2]
try:
    for qt in range(3):
        sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
        if sel:
            fitted_temps[qt] = fit_one_temp(sel)
    print("Temperaturas ajustadas (choice, score, noul):", [round(t, 3) for t in fitted_temps])
except Exception as e:
    print("Fallback na calibração:", e)

os.makedirs(OUTPUT_DIR, exist_ok=True)
save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}, os.path.join(OUTPUT_DIR, "model.safetensors"))
model.encoder.config.save_pretrained(os.path.join(OUTPUT_DIR, "encoder"))
tok.save_pretrained(os.path.join(OUTPUT_DIR, "tokenizer"))
cfg["fine_tuned"] = True
cfg["model_name"] = "laya-memorix-pt"
cfg["temperature"] = fitted_temps
cfg.pop("temperature_by_options", None)
with open(os.path.join(OUTPUT_DIR, "rl_agent_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)
print(f"Modelo salvo em {OUTPUT_DIR}")


## 6. Testar o modelo fine-tunado

In [ ]:
agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")
with open(DATASET_PATH, encoding="utf-8") as f:
    QUESTION = json.loads(f.readline())["questions"]  # mesma pergunta `choice` do treino
for texto in ["cria um plano de aula sobre revolução francesa",
              "apaga os flashcards que eu criei ontem",
              "explica fotossíntese e depois me faz umas questões"]:
    ans = agent_ft.predict(texto, QUESTION)["answers"]["geracao"]
    print(texto, "->", ans["choice"], {k: round(v, 3) for k, v in ans["probabilities"].items()})

# %% [markdown] cell:11b
## 6b. Avaliar no split de teste (pedidos que o treino nunca viu): baseline vs fine-tunado
Suba o `test.jsonl` gerado por `to_laya_jsonl.py` em `/content/test.jsonl` antes de rodar.
# %% [code] cell:12b
TEST_PATH = "/content/test.jsonl"
assert os.path.exists(TEST_PATH), f"Suba seu test.jsonl em {TEST_PATH} primeiro (painel Arquivos, à esquerda)"

test_rows = []
with open(TEST_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            test_rows.append(json.loads(line))
print(f"{len(test_rows)} pedidos de teste carregados")

def eval_agent(agent, rows, show_errors=False):
    correct = 0
    for row in rows:
        ans = agent.predict(row["state"], row["questions"])["answers"]["geracao"]
        gold = row["gold"]["geracao"]["probabilities"]
        ok = gold[ans["choice"]] > 0  # acerto se o top-1 é um dos rótulos do pedido
        correct += ok
        if show_errors and not ok:
            print(f"  ERRO: {row['state']!r} -> {ans['choice']} (gold {[k for k, v in gold.items() if v > 0]})")
    return correct / len(rows)

baseline_agent = laya.Agent(model_dir, device="cuda")  # checkpoint multilingual cru, pré fine-tuning
baseline_acc = eval_agent(baseline_agent, test_rows)
print(f"Baseline (sem fine-tuning): {baseline_acc:.3f} top-1 em {len(test_rows)} pedidos")

ft_acc = eval_agent(agent_ft, test_rows, show_errors=True)
print(f"Fine-tunado: {ft_acc:.3f} top-1 em {len(test_rows)} pedidos")
print(f"Delta: {ft_acc - baseline_acc:+.3f}")


## 7. (Opcional) baixar o modelo do Colab
`OUTPUT_DIR` some quando a sessão do Colab encerra. Baixe como zip ou suba pro seu Drive/HF Hub.

In [ ]:
import shutil
shutil.make_archive("/content/laya_finetuned", "zip", OUTPUT_DIR)
from google.colab import files
files.download("/content/laya_finetuned.zip")
